In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

In [ ]:

import scdrs
import scanpy as sc
sc.set_figure_params(dpi=125)
from anndata import AnnData
from scipy import stats
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import warnings

warnings.filterwarnings("ignore")

In [ ]:
cd /Users/niuruize/scDRS

In [ ]:
# subset gene sets
df_gs = pd.read_csv("/Users/niuruize/scDRS/EC/magma_10kb_top1000_zscore.74_traits.rv1.gs", sep="\t", index_col=0)

df_gs = df_gs.loc[
    [
        "PASS_ADHD_Demontis2018",
        "PASS_MDD_Howard2019",
        "PASS_Schizophrenia_Pardinas2018",
        "PASS_Alzheimers_Jansen2019",
        "PASS_BIP_Mullins2021",
        "PASS_Multiple_sclerosis",
        "PASS_Intelligence_SavageJansen2018",
        "UKB_460K.mental_NEUROTICISM",
        "PASS_VerbalNumericReasoning_Davies2018",
        "UKB_460K.body_BMIz",
        "UKB_460K.body_HEIGHTz", 
    ],
    :,
].rename(
    {
        "PASS_ADHD_Demontis2018": "ADHD",
        "PASS_MDD_Howard2019": "MDD",
        "PASS_Schizophrenia_Pardinas2018": "SCZ",
        "PASS_Alzheimers_Jansen2019": "AD",
        "PASS_BIP_Mullins2021": "BIP",
        "PASS_Multiple_sclerosis": "MS",
        "PASS_Intelligence_SavageJansen2018": "INT",
        "UKB_460K.mental_NEUROTICISM": "NRT",
        "PASS_VerbalNumericReasoning_Davies2018": "VNR",
        "UKB_460K.body_BMIz": "BMI",
        "UKB_460K.body_HEIGHTz": "Height",
    }
)
display(df_gs)

df_gs.to_csv("TS_HIP/processed_geneset.gs", sep="\t")

In [ ]:
%%capture

!scdrs compute-score \
    --h5ad-file TS_HIP/SC.h5ad \
    --h5ad-species human \
    --gs-file TS_HIP/processed_geneset.gs \
    --cov-file TS_HIP/SC_cov.tsv \
    --gs-species human \
    --flag-filter-data True \
    --flag-raw-count True \
    --flag-return-ctrl-raw-score False \
    --flag-return-ctrl-norm-score True \
    --out-folder TS_HIP/

In [ ]:
#
#sc.pp.normalize_total(adata, target_sum=1e4) 
#sc.pp.log1p(adata)
#sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
#sc.pl.highly_variable_genes(adata)
#adata = adata[:, adata.var.highly_variable]
#sc.pp.scale(adata, max_value=10)
#sc.tl.pca(adata, svd_solver='arpack')
#sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
#sc.tl.umap(adata)

In [ ]:
# load adata
adata = sc.read_h5ad("/Users/niuruize/scDRS/TS_HIP/HIP.h5ad")
cell_meta = pd.read_csv("/Users/niuruize/scDRS/TS_HIP/metadata.csv")

In [ ]:
with open("/Users/niuruize/scDRS/TS_HIP/gene_names.csv", 'r') as f:
  gene_names = f.read().splitlines()

In [ ]:
adata.obs = cell_meta
adata.obs.index = adata.obs['barcode']
adata.var.index = gene_names

In [ ]:
pca = pd.read_csv("/Users/niuruize/scDRS/TS_HIP/pca.csv")
pca.index = adata.obs.index

In [ ]:
adata.obsm['X_pca'] = pca.to_numpy()
adata.obsm['X_umap'] = np.vstack((adata.obs['UMAP_1'].to_numpy(), adata.obs['UMAP_2'].to_numpy())).T

In [ ]:
dict_score = {
    trait: pd.read_csv(f"TS_HIP/{trait}.full_score.gz", sep="\t", index_col=0)
    for trait in df_gs.index
}

for trait in dict_score:
    adata.obs[trait] = dict_score[trait]["norm_score"]

sc.set_figure_params(figsize=[6, 6], dpi=600)
sc.pl.umap(
    adata,
    color="celltype",
    ncols=1,
    color_map="RdBu_r",
    vmin=-5,
    vmax=5,
    save='HIP.pdf'
)
sc.set_figure_params(figsize=[10, 10], dpi=600)
sc.pl.umap(
    adata,
    color=dict_score.keys(),
    color_map="RdBu_r",
    vmin=-5,
    vmax=5,
    s=20,
    ncols=3,
    save='HIP_scDRS.pdf'
)

In [ ]:
adata.obs.Group_celltype

In [ ]:
%%capture

for trait in ["ADHD","MDD","SCZ","AD","BIP","MS","INT","NRT","VNR","BMI","Height"]:
    !scdrs perform-downstream \
        --h5ad-file TS_HIP/HIP.h5ad \
        --score-file TS_HIP/{trait}.full_score.gz \
        --out-folder TS_HIP/ \
        --group-analysis celltype \
        --flag-filter-data True \
        --flag-raw-count True

In [ ]:
# scDRS group-level statistics for SCZ
!cat TS_HIP/SCZ.scdrs_group.celltype | column -t -s $'\t'

In [ ]:
dict_df_stats = {
    trait: pd.read_csv(f"TS_HIP/{trait}.scdrs_group.celltype", sep="\t", index_col=0)
    for trait in ["ADHD","MDD","SCZ","AD","BIP","MS","INT","NRT","VNR","BMI","Height"]
}


fig, ax = scdrs.util.plot_group_stats(
    dict_df_stats={
        trait: df_stats.rename(index=adata.obs.celltype)
        for trait, df_stats in dict_df_stats.items()
    },
    plot_kws={
        "vmax": 0.1,
        "cb_fraction":0.12,
        "cb_vmax": 0.2
    }
)

plt.savefig("figures/HIP_3_celltype_scDRS.pdf", bbox_inches="tight")


In [ ]:
%%capture

for trait in ["ADHD","MDD","SCZ","AD","BIP","MS","INT","NRT","VNR","BMI","Height"]:
    !scdrs perform-downstream \
        --h5ad-file TS_HIP/HIP.h5ad \
        --score-file TS_HIP/{trait}.full_score.gz \
        --out-folder TS_HIP/ \
        --group-analysis Group_celltype \
        --flag-filter-data True \
        --flag-raw-count True

In [ ]:
dict_df_stats = {
    trait: pd.read_csv(f"TS_HIP/{trait}.scdrs_group.Group_celltype", sep="\t", index_col=0)
    for trait in ["ADHD","MDD","SCZ","AD","BIP","MS","INT","NRT","VNR","BMI","Height"]
}


fig, ax = scdrs.util.plot_group_stats(
    dict_df_stats={
        trait: df_stats.rename(index=adata.obs.celltype)
        for trait, df_stats in dict_df_stats.items()
    },
    plot_kws={
        "vmax": 0.1,
        "cb_fraction":0.12,
        "cb_vmax": 1.0
    }
)

plt.savefig("figures/HIP_1_group_celltype_scDRS.pdf", bbox_inches="tight")


In [ ]:
# extract subtype and perform a re-clustering
adata_Micro = adata[adata.obs["celltype"].isin(["Microglia"])].copy()
sc.pp.filter_cells(adata_Micro, min_genes=0)
sc.pp.filter_genes(adata_Micro, min_cells=1)
sc.pp.normalize_total(adata_Micro, target_sum=1e4)
sc.pp.log1p(adata_Micro)

sc.pp.highly_variable_genes(adata_Micro, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata_Micro = adata_Micro[:, adata_Micro.var.highly_variable]
sc.pp.scale(adata_Micro, max_value=10)
sc.tl.pca(adata_Micro, svd_solver="arpack")

sc.pp.neighbors(adata_Micro, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata_Micro, n_components=2)


In [ ]:
dict_score

In [ ]:
# assign scDRS score
dict_score = {
    trait: pd.read_csv(f"TS_HIP/{trait}.full_score.gz", sep="\t", index_col=0)
    for trait in ["MS","Height"]
}

for trait in dict_score:
    adata_Micro.obs[trait] = dict_score[trait]["norm_score"]
    
sc.pl.umap(
    adata_Micro,
    color=dict_score.keys(),
    color_map="RdBu_r",
    vmin=-5,
    vmax=5,
    s=20,
)

In [ ]:
#data.obsm['umap']=data.obsm['spatial']
#data.obs.refined_pred=data.obs.refined_pred.astype('str') # 数字转为字符串
Group=adata.obs['Group'].unique().tolist() # refined_pred是细胞注释
fig, axes = plt.subplots(3, 3, figsize=(18, 12), tight_layout=True,dpi=600) # nrows和ncols取决于想要的画图个数
x=0;y=0
for i in Group: 
    fig = sc.pl.umap(adata_Micro,color=dict_score.keys(),groups = i,color_map="RdBu_r",vmin=-5,vmax=5,s=20)
    x = x +1 if x <4 else 0
    y = y +1 if y <3 and x ==0 else y
plt.show()
#plt.savefig('./split.png')


In [ ]:
df_plot = adata_ca1.obs[["Dorsal", "SCZ", "Height"]].copy()
df_plot["Dorsal quintile"] = pd.qcut(df_plot["Dorsal"], 5, labels=np.arange(5))

fig, ax = plt.subplots(figsize=(3.5, 3.5))
for trait in ["SCZ", "Height"]:
    sns.lineplot(
        data=df_plot,
        x="Dorsal quintile",
        y=trait,
        label=trait,
        err_style="bars",
        marker="o",
        ax=ax,
    )
ax.set_xticks(np.arange(5))
ax.set_xlabel("Dorsal quintile")
ax.set_ylabel("Mean scDRS disease score")
fig.show()